#### setup

In [ ]:
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from library.data_utils import *
from library.models import *
from library.training import *
from library.evaluation import *
from library.rep_utils import *

In [ ]:
# device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(seed=0)

In [ ]:
# parameters
dataset = 'synsum'
train_frac = 0.8

#### data

In [ ]:
# load data and split
df = pd.read_csv(f"./data/datasets/{dataset}.csv", index_col=0)
train_df, val_df, test_df = make_splits(df=df, train_frac=train_frac, seed=0)

In [ ]:
# load embeddings
embeddings = torch.load(f"./data/embeddings/{dataset}.pt", weights_only=True)

In [ ]:
# set confounders
confounders_text = ['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high']
confounders_tabular = ['self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever']
confounders_full = confounders_text + confounders_tabular

#### train representation based model

In [ ]:
# set parameters
input_dim = len(confounders_tabular) + 768
params = dict(
    hidden_dim=64,
    learning_rate=5e-4,
    weight_decay=0,
    batch_size=128,
    max_epochs=50,
    patience=5,
    alpha=1.0)

In [ ]:
# init data loaders
train_loader, val_loader = make_tarnet_loaders(train_df, val_df, confounders_tabular, embeddings, batch_size=params['batch_size'])

In [ ]:
# init data loaders
train_loader, val_loader = make_tarnet_loaders(train_df, val_df, confounders_tabular, embeddings, batch_size=params['batch_size'])

# init and train model
tarnet = TARNet(input_dim, hidden_dim=params["hidden_dim"]).to(device)
tarnet, info = train_tarnet(tarnet, train_loader, val_loader, device,lr=params["learning_rate"], weight_decay=params["weight_decay"],
                            max_epochs=params["max_epochs"], patience=params["patience"], seed=0)

# store
torch.save(tarnet.state_dict(), './data/checkpoints/tarnet.pt')

In [ ]:
# init data loaders
train_loader, val_loader = make_dragonnet_loaders(train_df, val_df, confounders_tabular, embeddings, batch_size=params['batch_size'])

# init and train model
dragon = DragonNet(input_dim, hidden_dim=params["hidden_dim"]).to(device)
dragon, info = train_dragonnet(dragon, train_loader, val_loader, device,lr=params["learning_rate"], weight_decay=params["weight_decay"],
                            max_epochs=params["max_epochs"], patience=params["patience"], alpha=params["alpha"], seed=0)

# store
torch.save(dragon.state_dict(), './data/checkpoints/dragon.pt')

#### evaluation

In [ ]:
# set model to eval
tarnet.eval(); dragon.eval(); 

# init collectors
estimates_tarnet, estimates_dragon, cates, M0s, M1s = [], [], [], [], []

# init test loader
test_loader = DataLoader(EvalDataset(test_df, confounders_tabular, confounders_full, embeddings), batch_size=1024, shuffle=False)

In [ ]:
# loop over test loader
with torch.inference_mode():
    for phi, x, cate, M0, M1 in test_loader:

        # forward pass
        phi = phi.to(device)
        y0, y1 = tarnet(phi)
        tarnet_hats = y1 - y0
        e_logit, y0, y1 = dragon(phi)
        dragon_hats = y1 - y0

        # collect
        estimates_tarnet.append(tarnet_hats)
        estimates_dragon.append(dragon_hats)
        cates.append(cate.squeeze(-1))
        M0s.append(M0.squeeze(-1))
        M1s.append(M1.squeeze(-1))

# store
to_np = lambda parts: torch.cat(parts, dim=0).detach().cpu().numpy()
df_eval = pd.DataFrame({"tarnet": to_np(estimates_tarnet), "dragon": to_np(estimates_dragon), "cate": to_np(cates), "M0": to_np(M0s), "M1": to_np(M1s)})

In [ ]:
# sort and eval
df_eval['est'] = df_eval['tarnet']
ranked = df_eval.sort_values('tarnet', ascending=True).copy()
print(f"PEHE          : {pehe(ranked):.6f}")
print(f"Policy value  : {policy_value(ranked):.6f}")
print(f"AUTOC         : {autoc(ranked):.6f}")

In [ ]:
# sort and eval
df_eval['est'] = df_eval['dragon']
ranked = df_eval.sort_values('dragon', ascending=True).copy()
print(f"PEHE          : {pehe(ranked):.6f}")
print(f"Policy value  : {policy_value(ranked):.6f}")
print(f"AUTOC         : {autoc(ranked):.6f}")